# 5장 심화 실습 — 하남 GTFS 데이터 분석

기본 실습에서 정류장 세 곳의 시간표를 만든 뒤 진행합니다.
교재 5.6~5.7절의 선택 활동이며 기본 실습 제출물에는 포함하지 않습니다.
이 노트북은 하남 자료를 처음부터 읽습니다. 기본 노트북의 실행 상태는 필요하지 않습니다.

먼저 운행·방문 행 수를 구분하고, 정류장 한 곳의 다음 출발편을 조회합니다.
그다음 지역 선택, 지도, 패턴, 첫 출발 분포 중 관심 있는 활동을 이어서 진행합니다.
아래 준비 셀을 실행해 패키지 경로와 한글 글꼴을 설정합니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

import pandas as pd
import matplotlib.pyplot as plt
from lab import expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 운행별 시간표 조회 (교재 5.1~5.2)

`toy_feed()`는 A→B→C의 1호선 두 편과 D→E의 2호선 한 편을 담고 있습니다.
정류장 이름을 붙이기 전에 `trip_id`로 출발편 하나를 고르고 `stop_sequence`로 정렬합니다.

In [ ]:
from smartmob.teaching.raptor import toy_feed

small = toy_feed()
timetable = small["stop_times"].query("trip_id == 'L1-1'")
timetable = timetable.sort_values("stop_sequence")
timetable = timetable.merge(small["stops"], on="stop_id", validate="many_to_one")
timetable[["trip_id", "stop_sequence", "stop_name", "arrival_time", "departure_time"]]

A역 08:00, B역 08:10, C역 08:20 순서입니다. 같은 노선의 다음 출발편은 `L1-2`입니다.
`trip_id`를 바꾸어 실행하면 정류장 순서는 같고 시각은 30분 늦어집니다.

## 2. 데이터 규모와 교통수단 코드 (교재 5.6)

실습 표는 parquet로 저장되어 있습니다. `load_gtfs`는 표 이름을 키로 갖는 사전을 반환합니다.

In [ ]:
from smartmob.data import load_gtfs, KOREAN_ROUTE_TYPE

feed = load_gtfs("hanam")
pd.DataFrame([
    {"표": name, "행 수": len(table), "컬럼": ", ".join(table.columns)}
    for name, table in feed.items()
])

정류장 4,203개, 노선 169개, 운행 8,923개, 정류장 방문 656,385개, 운행일 규칙 1개입니다.
방문 행 수는 승객 수나 운행 편수가 아닙니다. 교통수단 코드별로 노선 수와 이름을 함께 봅니다.

In [ ]:
routes = feed["routes"].copy()
routes["route_type"] = routes["route_type"].astype(int)
types = routes.groupby("route_type").agg(
    노선수=("route_id", "size"),
    이름예시=("route_short_name", lambda s: ", ".join(s.astype(str).head(3))),
)
types["실습 코드 뜻"] = types.index.map(KOREAN_ROUTE_TYPE)
types

`0`은 이 파일의 코드표에서 시내·농어촌·마을버스이며 132개입니다.
GTFS 명세의 `0`은 트램이므로 공급자의 코드표를 먼저 확인합니다.
이 대응을 다른 GTFS에 그대로 적용하지 않습니다.

## 3. 운행일과 자정 이후 시각 (교재 5.3)

`calendar`의 요일별 값과 적용 기간을 확인합니다. 이 실습의 모든 운행이 같은 규칙을 쓰는지도 검사합니다.

In [ ]:
display(feed["calendar"])
print("운행에서 쓰는 service_id:", sorted(feed["trips"]["service_id"].unique()))
assert set(feed["trips"]["service_id"]) <= set(feed["calendar"]["service_id"])

`B1` 한 규칙에 모든 요일이 `1`로 들어 있습니다. 날짜별 예외 표는 제공하지 않습니다.
6장에서는 모든 운행을 한 운행일로 사용하며 질의 날짜를 자동으로 고르지 않습니다.
시각은 운행일을 기준으로 초로 바꿉니다.

In [ ]:
from smartmob.data import parse_gtfs_time, seconds_to_gtfs_time

for text in ["23:50:00", "24:20:00", "25:30:00"]:
    seconds = parse_gtfs_time(text)
    print(text, "→", seconds, "→", seconds_to_gtfs_time(seconds))
print("자정 통과 통행시간:",
      (parse_gtfs_time("24:20:00") - parse_gtfs_time("23:50:00")) // 60, "분")

`24:20:00`은 87,600초이고 자정을 통과한 통행시간은 30분입니다.
24시간으로 나눈 나머지만 남기면 운행일 다음 날이라는 정보를 잃습니다.
실제 파일에서 24시 이후의 방문 기록과 운행 첫 출발을 나누어 봅니다.

In [ ]:
st = feed["stop_times"].copy()
st["stop_sequence"] = st["stop_sequence"].astype(int)
st = st.sort_values(["trip_id", "stop_sequence"])
st["dep_s"] = st["departure_time"].map(parse_gtfs_time)
first = st.groupby("trip_id", sort=False).first()
print("24시 이상 방문 기록:", int((st["dep_s"] >= 86400).sum()))
print("방문 전체의 마지막 출발:", seconds_to_gtfs_time(st["dep_s"].max()))
print("운행 첫 정류장의 마지막 출발:", seconds_to_gtfs_time(first["dep_s"].max()))

24시 이상 방문은 22,560행입니다. 방문 전체의 마지막 출발은 `30:02:29`,
운행 첫 정류장의 마지막 출발은 `26:00:37`입니다. ‘막차’가 어느 정류장의 출발인지 구분합니다.

## 4. 정류장 식별과 운행 조회 (교재 5.6)

부분 문자열 검색으로 후보를 찾고 이름과 좌표를 확인합니다.

In [ ]:
stops = feed["stops"]
candidates = stops[stops["stop_name"].str.contains("하남시청", na=False)]
candidates[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

11개 후보에는 ‘하남시청소년수련관’도 들어 있습니다.
이번에는 버스 정류장 `BS_TAGO_GGB227000034` 하나를 선택합니다.
이름이 같은 다른 정류장은 좌표와 방향을 확인한 뒤 따로 조회합니다.

In [ ]:
target_stop_id = "BS_TAGO_GGB227000034"
visits = st[st["stop_id"] == target_stop_id].merge(
    feed["trips"][["trip_id", "route_id", "service_id"]],
    on="trip_id", validate="many_to_one",
).merge(
    routes[["route_id", "route_short_name", "route_type"]],
    on="route_id", validate="many_to_one",
)
ready = parse_gtfs_time("08:03:00")
next_departures = visits[visits["dep_s"] >= ready].sort_values("dep_s").head(10)
next_departures[["route_short_name", "trip_id", "departure_time", "stop_sequence"]]

선택한 정류장의 08:03 이후 출발 기록이 시각순으로 나옵니다.
서로 다른 노선의 운행이 섞여 있으며, 아직 목적지까지 가는 차인지 확인하지 않았습니다.
맨 앞 운행이 어디로 이어지는지 정류장 목록으로 읽습니다.

In [ ]:
next_trip_id = next_departures.iloc[0]["trip_id"]
next_trip = st[st["trip_id"] == next_trip_id].merge(
    stops[["stop_id", "stop_name"]], on="stop_id", validate="many_to_one",
)
board_sequence = next_departures.iloc[0]["stop_sequence"]
next_trip.loc[next_trip["stop_sequence"] >= board_sequence,
              ["stop_sequence", "stop_name", "departure_time"]].head(12)

승차 정류장 이후의 방문만 남겼습니다. 목적지가 이 목록에 없으면 다른 노선이나 환승도 살펴야 합니다.
경로를 자동으로 이어 보는 일은 6장에서 합니다.

## 5. 공간 범위에 따른 노선 선택 (교재 5.6)

하남 경계와 바깥 500m를 지나는 노선을 고릅니다.
정류장만 자르지 않고 선택된 노선의 전체 운행을 남기는 함수입니다.

In [ ]:
import geopandas as gpd
from smartmob.data import load_sigungu, clip_to_boundary

boundary = load_sigungu("하남시")
clipped = clip_to_boundary(feed, boundary, buffer_m=500)
pd.DataFrame({
    "선택 전": {name: len(feed[name]) for name in feed},
    "선택 후": {name: len(clipped[name]) for name in feed},
})

정류장은 4,203→4,128개, 노선은 169→167개입니다.
선택된 노선의 경계 밖 정류장을 남겨 두므로 정류장 수가 크게 줄지는 않습니다.
정류장 좌표와 시 경계를 같은 투영 좌표계로 그립니다.

In [ ]:
points = gpd.GeoDataFrame(
    stops, geometry=gpd.points_from_xy(stops["stop_lon"], stops["stop_lat"]), crs=4326,
).to_crs(5179)
area = gpd.GeoSeries([boundary], crs=4326).to_crs(5179)
kept = points[points["stop_id"].isin(clipped["stops"]["stop_id"])]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax in axes:
    points.plot(ax=ax, color="#bac5d1", markersize=3, label="실습 전체")
    kept.plot(ax=ax, color="#2869a4", markersize=3, label="선택된 노선의 정류장")
    area.boundary.plot(ax=ax, color="#b76417", linewidth=1.5)
    ax.set_axis_off()
axes[0].set_title("선택 노선의 정류장 전체")
xmin, ymin, xmax, ymax = area.total_bounds
axes[1].set_xlim(xmin - 1000, xmax + 1000)
axes[1].set_ylim(ymin - 1000, ymax + 1000)
axes[1].set_title("하남시 경계 부근 확대")
axes[0].legend(loc="upper right")
fig.tight_layout()

왼쪽은 자료 전체, 오른쪽은 하남 부근입니다. 주황색 선 밖에도 선택된 정류장이 남습니다.
이 선택은 경계 밖에서만 만나는 다른 노선으로의 환승까지 보존하지는 않습니다.
남긴 운행의 방문 기록을 중간에서 지우지 않았는지 확인합니다.

In [ ]:
expected = st[st["trip_id"].isin(clipped["trips"]["trip_id"])]
assert len(clipped["stop_times"]) == len(expected)
assert set(clipped["stop_times"]["stop_id"]) <= set(clipped["stops"]["stop_id"])
print("선택한 운행의 방문 기록과 정류장 연결을 유지했습니다.")

남긴 운행의 방문 수와 정류장 식별자가 맞습니다.
이는 표 관계 검사이며 모든 외부 환승 경로가 남았다는 검사는 아닙니다.

### 시간표와 노선도의 연계 탐색

노선과 운행을 바꾸며 시간표·정류장 위치를 비교합니다.
‘신규 노선 설계’에서는 기존 정류장을 선택하거나 도로 위에 정류장을 신설하고, 생성한 TXT 원문을 확인합니다.

In [ ]:
from smartmob.data import load_road_graph
from smartmob.viz.transit import gtfs_explorer

drive = load_road_graph("hanam", modes=("drive",))
gtfs_explorer(feed, boundary=boundary, road_graph=drive)

같은 노선에서 운행을 바꾸면 시간표가 달라집니다. 정류장 순서까지 달라지는지는 패턴 번호와 지도에서 확인합니다.
화면을 파일로 남기려면 반환값에 `.save("outputs/gtfs.html")`를 붙입니다.

## 6. 정류장 방문 순서에 따른 패턴 구성 (교재 5.7)

운행별 정류장 순서를 튜플로 만들고 노선 식별자와 함께 묶습니다.
상행·하행이나 일부 구간 운행을 시각표에서 구분하는 준비입니다.

In [ ]:
sequences = st.groupby("trip_id", sort=False)["stop_id"].agg(tuple).rename("stop_order")
trip_patterns = feed["trips"][["trip_id", "route_id"]].merge(
    sequences, on="trip_id", validate="one_to_one",
)
patterns = trip_patterns.groupby(["route_id", "stop_order"], sort=False).size().rename("운행 수")
patterns_per_route = patterns.groupby(level="route_id").size()
print("노선:", routes["route_id"].nunique(), "패턴:", len(patterns))
patterns_per_route.sort_values(ascending=False).head(8)

노선 169개가 패턴 349개로 나뉩니다. 패턴 수가 많은 노선이 곧 가장 자주 오는 노선은 아닙니다.
정류장 순서의 종류가 가장 많은 노선에서 몇 사례를 읽습니다.

In [ ]:
most_patterns_route = patterns_per_route.idxmax()
examples = trip_patterns[trip_patterns["route_id"] == most_patterns_route]
examples = examples.drop_duplicates("stop_order")
stop_names = stops.set_index("stop_id")["stop_name"]
for row in examples.head(4).itertuples():
    names = [stop_names[s] for s in row.stop_order]
    print(row.trip_id, len(names), "개:", " → ".join(names[:4]), "…", names[-1])

시작·끝 정류장과 정류장 수를 비교합니다. 패턴 수만 보고 상행·하행 때문이라고 단정하지 않습니다.
6장에서는 각 패턴에 운행별 시각 배열을 붙입니다.

## 7. 연습 — 시간대별 운행 시작 분포 ★★

앞서 만든 `first`는 운행당 한 행입니다. `dep_s // 3600`으로 출발 시간을 묶어 봅시다.
24시 이후도 24, 25, 26으로 남겨 운행일의 순서를 유지합니다.
`departures_by_hour`에 시간대별 운행 수를 넣으면 아래 셀이 그래프를 그립니다.

산출물: 시간대별 운행 수 그래프 1장, 가장 많은 시간대와 방문 기록을 세는 경우의 차이 2문장.

In [ ]:
departures_by_hour = None

todo("시간대별 첫 출발 운행 수", departures_by_hour)
if departures_by_hour is not None:
    assert departures_by_hour.sum() == len(feed["trips"])
    ax = departures_by_hour.sort_index().plot.bar(figsize=(10, 3), color="#2869a4")
    ax.set(xlabel="운행일 기준 첫 출발 시각(시)", ylabel="운행 수")
    plt.tight_layout()

합계는 8,923개 운행입니다. 656,385개라면 정류장 방문 횟수를 센 것은 아닌지 확인합니다.
이 그래프만으로 승객 수요나 특정 정류장의 배차간격을 알 수는 없습니다.

추가 연습 ★: `target_stop_id`를 다른 후보로 바꾸어 다음 출발편 10개를 비교해 봅시다.
산출물은 두 정류장의 이름·좌표와 출발편 표 2개입니다.